# 初始化LLM

In [1]:
from vllm import LLM, SamplingParams
from utils import load_jsonl
from typing import Iterable, List,Callable
from cs336_alignment.drgrpo_grader import r1_zero_reward_fn, question_only_reward_fn
from utils import *
from vllm_utils import init_vllm, evaluate_vllm

INFO 03-05 11:09:49 __init__.py:190] Automatically detected platform cuda.


编写一个脚本，用于评估 Qwen 2.5 Math 1.5B 在 MATH 上的零样本性能。该脚本应
+ （1）从 /data/a5-alignment/MATH/validation.jsonl 加载 MATH 验证示例，
+ （2）使用 r1_zero 提示将其格式化为语言模型的字符串提示，
+ （3）为每个示例生成输出。该脚本还应
+ （4）计算评估指标，
+ （5）将示例、模型生成内容以及相应的评估分数序列化到磁盘，以便在后续问题中进行分析。

In [2]:
# model_path ='checkpoints/checkpoint_499' # local path
model_path ='../model/Qwen2.5-Math-1.5B' # online path
llm = init_vllm(model_path,device='cuda',seed=42)
print('init LLM successfully!')

INFO 03-05 11:10:20 config.py:542] This model supports multiple tasks: {'score', 'reward', 'classify', 'embed', 'generate'}. Defaulting to 'generate'.
INFO 03-05 11:10:20 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.2) with config: model='../model/Qwen2.5-Math-1.5B', speculative_config=None, tokenizer='../model/Qwen2.5-Math-1.5B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=../model/Qwen2.5-Math-1.5B, num_

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 03-05 11:10:24 model_runner.py:1115] Loading model weights took 2.8797 GB
INFO 03-05 11:10:26 worker.py:267] Memory profiling takes 1.66 seconds
INFO 03-05 11:10:26 worker.py:267] the current vLLM instance can use total_gpu_memory (23.58GiB) x gpu_memory_utilization (0.85) = 20.05GiB
INFO 03-05 11:10:26 worker.py:267] model weights take 2.88GiB; non_torch_memory takes 0.06GiB; PyTorch activation peak memory takes 1.40GiB; the rest of the memory reserved for KV Cache is 15.71GiB.
INFO 03-05 11:10:26 executor_base.py:110] # CUDA blocks: 36777, # CPU blocks: 9362
INFO 03-05 11:10:26 executor_base.py:115] Maximum concurrency for 4096 tokens per request: 143.66x
INFO 03-05 11:10:33 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_u

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:26<00:00,  1.34it/s]

INFO 03-05 11:10:59 model_runner.py:1562] Graph capturing finished in 26 secs, took 0.20 GiB
INFO 03-05 11:10:59 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 35.46 seconds


init LLM successfully!


In [5]:
# load_template
template_path = '../cs336_alignment/prompts/r1_zero.prompt'
r1_template = load_template(template_path)

# generation
import os 
json_path  = '../preprocessed/math/train.jsonl'
prompts = get_r1_prompts(json_path,r1_template)
ground_truths = get_r1_ground_truths(json_path)# only answer
ground_truths_with_template = get_r1_ground_truths_with_template(json_path)#) # with template

sampling_params = SamplingParams(
    temperature=1.0,
    top_p=1.0,
    max_tokens=1024,
    stop=["</answer>"],
    include_stop_str_in_output = True # in need
)
prompts[1],ground_truths[1],ground_truths_with_template[1]

('A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: If $5x - 3 = 12$, what is the value of $5x + 3$?\nAssistant: <think>',
 '18',
 'Adding 6 to both sides of $5x - 3 =12$ gives $5x -3 + 6 = 12 + 6$.  Simplifying both sides gives $5x + 3 = \\boxed{18}$. </think> <answer> 18 </answer>')

In [ ]:
evaluate_vllm(
    llm,
    prompts,
    ground_truths,
    sampling_params,
    r1_zero_reward_fn    
)

Processed prompts: 100%|██████████| 12000/12000 [23:15<00:00,  8.60it/s, est. speed input: 1426.97 toks/s, output: 3116.51 toks/s] 


{'sample_size': 12000,
 'answer_correct': 330,
 'format_correct': 2037,
 'total_correct': 330,
 'format_correct_but_answer_wrong': 1707,
 'answer_correct_but_format_wrong': 0,
 'total_wrong': 9963,
 'accuracy': 0.0275,
 'wrong_rate': 0.83025,
 'contradictory_samples': 0}

: 

提示词
A conversation between User and Assistant. 
The User asks a question, and the Assistant solves it. 
The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. 
The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>

### outputs: List[RequestOutput]
+ output.request_id           # 请求ID
+ output.prompt               # 原始输入的prompt
+ output.prompt_token_ids     # prompt的token IDs
+ output.outputs              # 生成的输出列表（可能多个，如果设置n>1）
+ output.finished             # 是否完成生成

In [7]:
outputs = llm.generate([r1_prompt_example],sampling_params)
outputs[0].prompt

Processed prompts: 100%|██████████| 1/1 [00:06<00:00,  6.12s/it, est. speed input: 33.33 toks/s, output: 167.29 toks/s]


'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Define\n\\[p = \\sum_{k = 1}^\\infty \\frac{1}{k^2} \\quad \\text{and} \\quad q = \\sum_{k = 1}^\\infty \\frac{1}{k^3}.\\]Find a way to write\n\\[\\sum_{j = 1}^\\infty \\sum_{k = 1}^\\infty \\frac{1}{(j + k)^3}\\]in terms of $p$ and $q.$\nAssistant: <think>'

In [8]:
outputs[0].outputs[0].text

" We note that the sum is similar to the Legendre triplets representation. A Legendre triplet is a triple of positive integers $a, b, c$ such that $a, b, c$ are pairwise coprime and are the coefficients of a quadratic form in three variables which represents all positive integers except one. The standard form of a Legendre triplet is:\n\\[\\ell(n) = an^2 + bn + c\\]\nwhere $a, b, c \\in \\mathbb{Z}$ and $\\gcd(a, b, c) = 1.$\nFor example, \\( (3, 4, 5) = \\frac{29n^2 + 38n + 17}{4} \\)\nis a Legendre triplet. \\\\\nSee also:\n\\[\\sum_{n=1}^\\infty \\frac{1}{(\\ell(n))^2} = \\frac{\\pi^2}{4\\varphi^2} + \\frac{6}{\\varphi^3}\\]\nand\n\\[\\sum_{n=1}^\\infty \\frac{1}{(\\ell(n))^3} = \\frac{9\\pi^3}{240}\\]\nThe sums are convergent, and they have closed forms. \\\\\nThe problem requires us to find a way to express the sum\n\\[\\sum_{j = 1}^\\infty \\sum_{k = 1}^\\infty \\frac{1}{(j + k)^3}\\]\nin terms of $p$ and $q$, where\n\\[p = \\sum_{k = 1}^\\infty \\frac{1}{k^2} \\quad \\text{and} 

# 正式定义evaluate_vllm函数
+ 奖励函数: r1_zero_reward_fn, question_only_reward_fn

In [9]:
# 获得prompts
# 将jsonl读取的迭代器转换为列表,以便batched
json_path ='../preprocessed/math/test.jsonl'
json_iter = load_jsonl(json_path)
prompts = [apply_r1_template(prompt_template,json_obj) for json_obj in json_iter]

# ground_truth不包含模板内容

In [ ]:
# 获得ground_truth 
# json_path ='../preprocessed/math/test.jsonl'
# json_iter = load_jsonl(json_path)
# ground_truth_template = "{cot}</think><answer>{answer}</answer>"# <think>已经出现在prompt中
# def apply_ground_truth_template(
#     prompt:str,
#     json_obj:dict
# )->str:
#     return prompt.format(cot=json_obj['cot'],answer=json_obj['answer'])
# ground_truths = [apply_ground_truth_template(ground_truth_template,json_obj) for json_obj in json_iter]

In [17]:
json_path ='../preprocessed/math/test.jsonl'
json_iter = load_jsonl(json_path)
ground_truths= [json_obj['answer'] for json_obj in json_iter]
ground_truths[:5]

['\\left( 3, \\frac{\\pi}{2} \\right)',
 'p - q',
 '\\frac{14}{3}',
 '9',
 '\\text{Evelyn}']

In [22]:
from typing import List,Callable
from cs336_alignment.drgrpo_grader import r1_zero_reward_fn, question_only_reward_fn
def evaluate_vllm(
    llm:LLM,
    sampling_params:SamplingParams,
    prompts:List[str],
    reward_fn:Callable,# defined in drgrpo_grader.py 
)->None:
    outputs = llm.generate(prompts,sampling_params)
    rewards = [reward_fn(output.outputs[0].text, ground_truth) for output,ground_truth in zip(outputs,ground_truths)]
    return outputs, rewards
outputs, rewards = evaluate_vllm(llm,sampling_params,prompts,r1_zero_reward_fn)

Processed prompts: 100%|██████████| 500/500 [00:40<00:00, 12.50it/s, est. speed input: 2051.17 toks/s, output: 4676.06 toks/s]


In [23]:
prompts[6],ground_truths[6],outputs[6].outputs[0].text,rewards[6]

('A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: What is the smallest positive perfect cube that can be written as the sum of three consecutive integers?\nAssistant: <think>',
 '27',
 ' $1^3 = 1*1*1 = 1$\n$2^3=1+2+3=3^3 = 3*3*3 = 27$\n$3^3= 1+2+3+4+5+6 = 21$\n4^3 = 1 + 2 + 3 +......... + 7 + 8 + 9 + 10 + 11 + 12 + 13 + 14 + 15 + 16 = 169\nAnswer: 8 is answer.? </think> <answer> ? </answer>',
 {'format_reward': 1.0, 'answer_reward': 0.0, 'reward': 0.0})

# 可视化奖励

In [24]:
from typing import List, Dict, Tuple, Union
import pandas as pd
import numpy as np
import random
def analyze_rewards(rewards: List[Dict[str, float]]) -> Dict[str, Union[float, int, str]]:
    """分析reward数据，统计各项指标"""
    total = len(rewards)
    if total == 0:
        return {}
    
    # 统计各类别
    format_1_answer_1 = sum(1 for r in rewards if r['format_reward']==1 and r['answer_reward']==1)
    format_1_answer_0 = sum(1 for r in rewards if r['format_reward']==1 and r['answer_reward']==0)
    format_0_answer_0 = sum(1 for r in rewards if r['format_reward']==0 and r['answer_reward']==0)
    format_0_answer_1 = sum(1 for r in rewards if r['format_reward']==0 and r['answer_reward']==1)
    
    return {
        'total': total,
        'correct_both': format_1_answer_1, 'pct_both': f"{format_1_answer_1/total:.2%}",
        'format_only': format_1_answer_0, 'pct_format_only': f"{format_1_answer_0/total:.2%}",
        'wrong_format': format_0_answer_0, 'pct_wrong_format': f"{format_0_answer_0/total:.2%}",
        'answer_only': format_0_answer_1, 'pct_answer_only': f"{format_0_answer_1/total:.2%}",
    }
def visualize_rewards(rewards: List[Dict[str, float]], title: str = "Reward Analysis") -> pd.DataFrame:
    """使用pandas可视化reward统计结果"""
    stats = analyze_rewards(rewards)
    if not stats:
        return pd.DataFrame()
    
    # 创建DataFrame
    df_summary = pd.DataFrame([{
        'Category': 'Both Correct (F=1,A=1)', 'Count': stats['correct_both'], 'Percentage': stats['pct_both'],
        'Format': 1, 'Answer': 1
    }, {
        'Category': 'Format Only (F=1,A=0)', 'Count': stats['format_only'], 'Percentage': stats['pct_format_only'],
        'Format': 1, 'Answer': 0
    }, {
        'Category': 'Wrong Format (F=0,A=0)', 'Count': stats['wrong_format'], 'Percentage': stats['pct_wrong_format'],
        'Format': 0, 'Answer': 0
    }, {
        'Category': 'Answer Only (F=0,A=1)', 'Count': stats['answer_only'], 'Percentage': stats['pct_answer_only'],
        'Format': 0, 'Answer': 1
    }])
    
    # # 打印统计信息
    # print(f"\n{'='*60}\n{title:^60}\n{'='*60}")
    # print(f"总样本数: {stats['total']}")
    # print(df_summary[['Category', 'Count', 'Percentage']].to_string(index=False))
    
    # # 创建详细数据透视表
    # df_raw = pd.DataFrame(rewards)
    # pivot = pd.crosstab(df_raw['format_reward'], df_raw['answer_reward'], 
    #                     margins=True, margins_name='Total')
    # pivot.index = ['Format=0', 'Format=1', 'Total']
    # pivot.columns = ['Answer=0', 'Answer=1', 'Total']
    
    # print(f"\n{'='*60}\n混淆矩阵:\n{'='*60}")
    # print(pivot)
    
    return df_summary

def compare_experiments(experiments: Dict[str, List[Dict[str, float]]]) -> pd.DataFrame:
    """比较多个实验的结果"""
    results = []
    for name, rewards in experiments.items():
        stats = analyze_rewards(rewards)
        results.append({
            'Experiment': name,
            'Total': stats['total'],
            'Both Correct': f"{stats['correct_both']} ({stats['pct_both']})",
            'Format Only': f"{stats['format_only']} ({stats['pct_format_only']})",
            'Wrong Format': f"{stats['wrong_format']} ({stats['pct_wrong_format']})",
            'Answer Only': f"{stats['answer_only']} ({stats['pct_answer_only']})",
        })
    
    df_compare = pd.DataFrame(results)
    print(f"\n{'='*80}\n实验对比结果\n{'='*80}")
    print(df_compare.to_string(index=False))
    return df_compare

In [25]:
visualize_rewards(rewards, title= 'R1 zero shot reward')

,Category,Count,Percentage,Format,Answer
0,Both Correct (F=...,13,2.60%,1,1
1,Format Only (F=1...,77,15.40%,1,0
2,Wrong Format (F=...,410,82.00%,0,0
3,Answer Only (F=0...,0,0.00%,0,1
